In [1]:
import os

from dotenv import load_dotenv

from hakken_agents.config import DocumentConfig, LLMConfig
from hakken_agents.graph_builder.enki.extract_entities import ExtractEntities
from hakken_agents.utils.file import load_file

load_dotenv()

True

In [2]:
ROOT_FOLDER = "/Users/Pablo.Sanchez2/Documents/GitHub/project_spaice_ds/packages/pip/hakken-agents"
PROMPTS_FOLDER = f"{ROOT_FOLDER}/prompts/graph_builder"

In [ ]:
doc_config = DocumentConfig(
    path=f"{ROOT_FOLDER}/documents/paris.txt",
    encoding="utf-8",
    group_id="test",
    source_description="This is a document about Paris",
    reference_year=2020,
    source_type="text",
)
llm_config = LLMConfig(name="upstage/solar-pro-3:free")
print(llm_config.model_dump_json(indent=4))

In [ ]:
content = load_file(doc_config.path, doc_config.encoding)
print(f"{content[:100]}...")
content_i = content[:400]

In [ ]:
from langchain_core.prompts import PromptTemplate

system_prompt = load_file(f"{PROMPTS_FOLDER}/extract_entities/system_v0.txt", "utf-8")
user_prompt_template = PromptTemplate.from_file(
    template_file=f"{PROMPTS_FOLDER}/extract_entities/user_v0.txt", template_format="jinja2"
)

In [ ]:
extract_entities = ExtractEntities(system_prompt, user_prompt_template, llm_config)

In [ ]:
from langchain_core.documents import Document
from langgraph.constants import START
from langgraph.graph import END, StateGraph

from hakken_agents.graph_builder.enki.state import EnkiState

# Build the graph
workflow = StateGraph(EnkiState)
workflow.add_node("extract", extract_entities)
workflow.add_edge(START, "extract")
workflow.add_edge("extract", END)

app = workflow.compile()

In [ ]:
initial_state: EnkiState = {
    "document": Document(page_content=content_i),
    "extracted_entities": [],
    "num_new_domains": 0,
    "num_new_entities": 0,
    "num_new_relations": 0,
    "extracted_facts": [],
}

result = app.invoke(initial_state)

In [ ]:
result

In [3]:
from hakken_agents.config import EmbedderConfig
from hakken_agents.db.config import PostgresDBConfig
from hakken_agents.vector_db.config import VectorDBConfig, VectorDBTableConfig
from hakken_agents.vector_db.engine import VectorDBEngine

db_config = PostgresDBConfig()

embeddings_config = EmbedderConfig(
    api_key=os.getenv("OPENAI_API_KEY"), embedding_model="text-embedding-3-small"
)

table_config = VectorDBTableConfig(
    name="entities_vectors",
    schema_name="public",
    content_column="content",
    embedding_column="embedding",
    metadata_columns=["domain_id", "domain", "name", "description"],
)
vector_db_config = VectorDBConfig(
    db=db_config,
    embedder=embeddings_config,
    table=table_config,
)

print(db_config)

user='postgres' password='postgres' host='localhost' port=5432 database='hakken_agents'


In [ ]:
vector_db = await VectorDBEngine.create_from_config(vector_db_config)

2026-01-28 18:56:02.168 | WARNING  | hakken_agents.vector_db.engine:ainit_vectorstore_table:67 - Table entities_vectors already exists in schema public. Skipping initialization.


In [5]:
count = await vector_db.acount_documents(table_name="entities_vectors", schema_name="public")
print(count)
docs = await vector_db.alist_documents(
    table_name="entities_vectors", schema_name="public", id_column="langchain_id"
)

8


In [6]:
for doc in docs:
    print(doc)

page_content='BRCA1 (human_gene):
A human tumor suppressor gene involved in DNA repair. Mutations are associated with increased risk of breast and ovarian cancer.' metadata={'domain': 'human_gene', 'name': 'BRCA1', 'description': 'A human tumor suppressor gene involved in DNA repair. Mutations are associated with increased risk of breast and ovarian cancer.'}
page_content='BRCA1 (human_gene):
A human gene that encodes a protein crucial for maintaining genomic stability by helping repair damaged DNA, especially double-strand breaks.' metadata={'domain': 'human_gene', 'name': 'BRCA1', 'description': 'A human gene that encodes a protein crucial for maintaining genomic stability by helping repair damaged DNA, especially double-strand breaks.'}
page_content='Paris (city):
Paris is the capital of France' metadata={'domain': 'city', 'name': 'Paris', 'description': 'Paris is the capital of France'}
page_content='bank (animal_group):
A large group of fish swimming together, also called a shoal.

In [7]:
docs[0]

Document(id='2cd431db-690f-430a-a9e5-c9c3e86d34cc', metadata={'domain': 'human_gene', 'name': 'BRCA1', 'description': 'A human tumor suppressor gene involved in DNA repair. Mutations are associated with increased risk of breast and ovarian cancer.'}, page_content='BRCA1 (human_gene):\nA human tumor suppressor gene involved in DNA repair. Mutations are associated with increased risk of breast and ovarian cancer.')

In [8]:
from hakken_agents.graph_builder.schemas.entity import EntityDB

entities = [EntityDB.from_document(doc) for doc in docs]

In [ ]:
from hakken_agents.graph_builder.enki.dedup_entities.embeddings import DeduplicateEntities
from hakken_agents.graph_builder.enki.state import EnkiContextSchema, EnkiStateHelper

dedup_entities = DeduplicateEntities(vector_db=vector_db)

In [ ]:
entities

In [ ]:
state = {}
from langgraph.runtime import Runtime

EnkiStateHelper.set_extracted_entities(state, entities)
print(state)

print(f"Initial entities: {len(entities)}")

runtime = Runtime(context=EnkiContextSchema(dedup_entities_threshold=0.85))
state = await dedup_entities(state, runtime=runtime)
new_entities = EnkiStateHelper.get_extracted_entities(state)
print(f"New entities: {len(new_entities)}")

In [ ]:
state

In [9]:
import numpy as np

from hakken_agents.vector_db.enums import SimilarityMetric

query_list = [doc.page_content for doc in docs]
simlarit_matrix = await vector_db.asimilarity_scores(query_list, metric=SimilarityMetric.COSINE)
# Print with 2 decimal places
np.array(simlarit_matrix).round(2)

array([[1.  , 0.86, 0.05, 0.08, 0.78, 0.03, 0.07, 0.02],
       [0.86, 1.  , 0.06, 0.09, 0.84, 0.07, 0.09, 0.06],
       [0.05, 0.06, 1.  , 0.11, 0.06, 0.62, 0.17, 0.11],
       [0.08, 0.09, 0.11, 1.  , 0.21, 0.14, 0.37, 0.06],
       [0.78, 0.84, 0.06, 0.21, 1.  , 0.08, 0.12, 0.02],
       [0.03, 0.07, 0.62, 0.14, 0.08, 1.  , 0.19, 0.11],
       [0.07, 0.09, 0.17, 0.37, 0.12, 0.19, 1.  , 0.13],
       [0.02, 0.06, 0.11, 0.06, 0.02, 0.11, 0.13, 1.  ]])

In [ ]:
entity_list = [
    EntityDB(
        name="Madrid",
        description="Madrid is the capital of Spain",
        domain_id="1",
        domain="city",
    ),
    EntityDB(
        name="Paris",
        description="Paris is the capital of France",
        domain_id="2",
        domain="city",
    ),
    EntityDB(
        name="temperature",
        description="The degree or intensity of heat present in a substance or object",
        domain_id="3",
        domain="physical_quantity",
    ),
    EntityDB(
        name="bank",
        description="A large group of fish swimming together, also called a shoal.",
        domain_id="4",
        domain="animal_group",
    ),
    EntityDB(
        name="bank",
        description="A financial institution that accepts deposits, lends money, and provides other financial services.",
        domain_id="5",
        domain="financial_institution",
    ),
    EntityDB(
        name="BRCA1",
        description="A human tumor suppressor gene involved in DNA repair. Mutations are associated with increased risk of breast and ovarian cancer.",
        domain_id="6",
        domain="human_gene",
    ),
    EntityDB(
        name="BRCA1",
        description="An animal gene homolog of BRCA1 involved in DNA repair and maintaining genome stability (e.g., in mouse and other mammals).",
        domain_id="7",
        domain="animal_gene",
    ),
    EntityDB(
        name="BRCA1",
        description="A human gene that encodes a protein crucial for maintaining genomic stability by helping repair damaged DNA, especially double-strand breaks.",
        domain_id="8",
        domain="human_gene",
    ),
]


docs = [entity.to_document() for entity in entity_list]

await vector_db.aadd_documents(docs)

In [ ]:
docs = await vector_db._store.asimilarity_search("", k=10_000)

In [ ]:
docs[0]